In [50]:
import jupedsim as jps 
from shapely import Polygon
import pathlib
import numpy as np
trajectory_file = pathlib.Path("simple_evac.sqlite")
if trajectory_file.exists():
    trajectory_file.unlink()
room = Polygon([(0, 0), (10, 0), (10, 8), (0, 8)])


exit_right = Polygon([
    (9.5, 3),
    (10, 3),
    (10, 5),
    (9.5, 5)
])
exit_left = Polygon([
    (0, 3),
    (0.5, 3),
    (0.5, 5),
    (0, 5)
])
start_positions = [(x,y) for x in range(3,8) for y in range(3,8)]

def create_simulation(policy,seed=1,record=False):
    rng = np.random.default_rng(seed)
    
    if record:
        writer =jps.SqliteTrajectoryWriter(
            output_file=pathlib.Path("simple_evac.sqlite"),
            every_nth_frame=5)
    else:
        writer = None
    simulation = jps.Simulation(
        model=jps.CollisionFreeSpeedModel(),
        geometry=room,
        trajectory_writer= writer
        
    )
    exit_id_right = simulation.add_exit_stage(exit_right)
    exit_id_left = simulation.add_exit_stage(exit_left)


    journey_left = jps.JourneyDescription([exit_id_left])
    journey_right = jps.JourneyDescription([exit_id_right])
    journey_id_left = simulation.add_journey(journey_left)
    journey_id_right = simulation.add_journey(journey_right)
    left = (journey_id_left,exit_id_left)
    right = (journey_id_right,exit_id_right)

    for pos in start_positions:
        route = choose_route(pos,policy,left,right,rng)
        parameters = jps.CollisionFreeSpeedModelAgentParameters(position=pos,journey_id=route[0],stage_id=route[1])
        simulation.add_agent(parameters)
    return simulation

def choose_route(position,policy,left,right,rng):
    if policy == 'left':
        return left
    elif policy == 'right':
        return right
    elif policy == 'random':
        if rng.random()<0.5:
            return left
        else:
            return right
    elif policy == 'spatial':
        if position[0] >5:
            return right
        elif position[0]<5:
            return left
        else:
            if rng.random()<0.5:
                return left
            else:
                return right
                    
    


steps = 0
simulation = create_simulation('random',record=True)
try:
    while simulation.agent_count() >0:
        simulation.iterate()
        steps+=1
finally:
    simulation._writer.close()
print(f"steps {steps}, time: {steps*simulation.delta_time()}, agent count {simulation.agent_count()}")


steps 877, time: 8.77, agent count 0


In [40]:
from jupedsim.internal.notebook_utils import animate, read_sqlite_file
trajectory_data, walkable_area = read_sqlite_file("simple_evac.sqlite")

animation = animate(
    trajectory_data,
    walkable_area,
    every_nth_frame=2
)

animation.show()

In [52]:
policies = ["left", 'right','random','spatial']
episodeNum=100
for policy in policies:
    episodeSteps = []
    for i in range(episodeNum):
        simulation = create_simulation(policy=policy,seed=i)
        steps = 0    
        while simulation.agent_count() >0:
            simulation.iterate()
            steps+=1
        episodeSteps.append(steps*simulation.delta_time())
    print(f'Num of episodes: {episodeNum}| policy: {policy}| average time: {np.mean(episodeSteps)}|stDeviation {np.std(episodeSteps)} | min {np.min(episodeSteps)}| max {np.max(episodeSteps)}')
            

Num of episodes: 100| policy: left| average time: 9.549999999999997|stDeviation 3.552713678800501e-15 | min 9.55| max 9.55
Num of episodes: 100| policy: right| average time: 9.549999999999997|stDeviation 3.552713678800501e-15 | min 9.55| max 9.55
Num of episodes: 100| policy: random| average time: 9.065000000000001|stDeviation 0.6771122506645408 | min 7.29| max 10.6
Num of episodes: 100| policy: spatial| average time: 6.170099999999999|stDeviation 0.48991528859589606 | min 5.74| max 7.640000000000001
